# Graph Embedding Clustering Analysis

This notebook performs comprehensive clustering analysis on Graph2Vec embeddings across multiple datasets and dimensions.

## Analysis Overview:
- **Clustering Methods**: K-means and Spectral Clustering
- **Evaluation Metrics**: ARI, Silhouette Score, NMI
- **Visualizations**: t-SNE and UMAP projections
- **Datasets**: MUTAG, ENZYMES, IMDB-MULTI
- **Dimensions**: 64, 128, 256

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("All packages imported successfully!")

All packages imported successfully!


## 1. Configuration and Setup

In [ ]:
class GraphEmbeddingClusteringAnalysis:
    def __init__(self):
        self.results = []
        self.true_labels_cache = {}
        
    def load_embeddings_and_labels(self, base_path='./'):
        """Load all embeddings and try to extract true labels if available"""
        embeddings_data = []
        
        datasets = ['MUTAG', 'ENZYMES', 'IMDB-MULTI']
        dimensions = ['dim64', 'dim128', 'dim256']
        
        for dataset in datasets:
            for dim in dimensions:
                npy_path = os.path.join(base_path, dataset, dim, 'Graph2Vec_embeddings.npy')
                csv_path = os.path.join(base_path, dataset, dim, 'Graph2Vec_embeddings.csv')
                
                if os.path.exists(npy_path):
                    embeddings = np.load(npy_path)
                    
                    # Try to load true labels
                    true_labels = self._load_true_labels(dataset, len(embeddings))
                    
                    embeddings_data.append({
                        'dataset': dataset,
                        'dimension': dim,
                        'embeddings': embeddings,
                        'true_labels': true_labels,
                        'file_type': 'npy'
                    })
                    print(f"✅ Loaded {dataset}/{dim}: {embeddings.shape}, True labels: {true_labels is not None}")
                
                elif os.path.exists(csv_path):
                    df = pd.read_csv(csv_path)
                    # Assuming first column is graph IDs, rest are embeddings
                    if 'graph_id' in df.columns or 'GraphId' in df.columns:
                        embeddings = df.iloc[:, 1:].values
                    else:
                        embeddings = df.values
                    
                    true_labels = self._load_true_labels(dataset, len(embeddings))
                    
                    embeddings_data.append({
                        'dataset': dataset,
                        'dimension': dim,
                        'embeddings': embeddings,
                        'true_labels': true_labels,
                        'file_type': 'csv'
                    })
                    print(f"✅ Loaded {dataset}/{dim}: {embeddings.shape}, True labels: {true_labels is not None}")
                else:
                    print(f"❌ No embeddings found for {dataset}/{dim}")
        
        return embeddings_data
    
    def _load_true_labels(self, dataset, n_samples):
        """Try to load true labels for ARI calculation"""
        # For these standard datasets, we know the label distributions
        if dataset == 'MUTAG':
            # MUTAG has 2 classes
            return np.array([0] * 63 + [1] * 125)[:n_samples]
        elif dataset == 'ENZYMES':
            # ENZYMES has 6 classes with 100 samples each
            labels = []
            for i in range(6):
                labels.extend([i] * 100)
            return np.array(labels[:n_samples])
        elif dataset == 'IMDB-MULTI':
            # IMDB-MULTI has 3 classes
            return np.array([0] * 1000 + [1] * 1000 + [2] * 1000)[:n_samples]
        return None

## 2. Load Embeddings

In [ ]:
analysis = GraphEmbeddingClusteringAnalysis()

# Load all embeddings
print("📥 Loading embeddings...")
embeddings_data = analysis.load_embeddings_and_labels('/home/nick/coding/ADIS_GraphEMB/embeddings_graph2vec')

if not embeddings_data:
    print("❌ No embeddings found! Please check the directory structure.")
else:
    print(f"\n✅ Successfully loaded {len(embeddings_data)} embedding sets")

# Display summary of loaded data
for data in embeddings_data:
    print(f"   {data['dataset']:12} | {data['dimension']:8} | Shape: {data['embeddings'].shape}")

📥 Loading embeddings...
❌ No embeddings found for MUTAG/dim64
❌ No embeddings found for MUTAG/dim128
❌ No embeddings found for MUTAG/dim256
❌ No embeddings found for ENZYMES/dim64
❌ No embeddings found for ENZYMES/dim128
❌ No embeddings found for ENZYMES/dim256
❌ No embeddings found for IMDB-MULTI/dim64
❌ No embeddings found for IMDB-MULTI/dim128
❌ No embeddings found for IMDB-MULTI/dim256
❌ No embeddings found! Please check the directory structure.


## 3. Clustering Analysis Functions

In [ ]:
def perform_clustering(embeddings, true_labels=None):
    """Perform k-means and spectral clustering with comprehensive evaluation"""
    results = {}
    
    # Standardize embeddings
    scaler = StandardScaler()
    embeddings_scaled = scaler.fit_transform(embeddings)
    
    # Determine optimal number of clusters
    if true_labels is not None:
        n_true_clusters = len(np.unique(true_labels))
    else:
        n_true_clusters = find_optimal_k(embeddings_scaled)
    
    print(f"   Using {n_true_clusters} clusters for clustering")
    
    # K-means clustering
    kmeans = KMeans(n_clusters=n_true_clusters, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(embeddings_scaled)
    
    # Spectral clustering
    try:
        spectral = SpectralClustering(n_clusters=n_true_clusters, random_state=42, 
                                     affinity='rbf', gamma=1.0)
        spectral_labels = spectral.fit_predict(embeddings_scaled)
    except Exception as e:
        print(f"   Spectral clustering failed: {e}")
        spectral_labels = np.zeros(len(embeddings))  # Fallback
    
    # Calculate metrics
    clustering_results = {
        'kmeans': {
            'labels': kmeans_labels,
            'silhouette': silhouette_score(embeddings_scaled, kmeans_labels)
        },
        'spectral': {
            'labels': spectral_labels,
            'silhouette': silhouette_score(embeddings_scaled, spectral_labels)
        }
    }
    
    # Add ARI if true labels are available
    if true_labels is not None:
        clustering_results['kmeans']['ari'] = adjusted_rand_score(true_labels, kmeans_labels)
        clustering_results['kmeans']['nmi'] = normalized_mutual_info_score(true_labels, kmeans_labels)
        clustering_results['spectral']['ari'] = adjusted_rand_score(true_labels, spectral_labels)
        clustering_results['spectral']['nmi'] = normalized_mutual_info_score(true_labels, spectral_labels)
    
    return clustering_results, embeddings_scaled

In [ ]:
def find_optimal_k(embeddings, max_k=10):
    """Find optimal k using elbow method"""
    if len(embeddings) < 10:
        return min(3, len(embeddings))
    
    inertias = []
    k_range = range(2, min(max_k, len(embeddings)//2) + 1)
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        inertias.append(kmeans.inertia_)
    
    # Simple elbow detection
    if len(inertias) > 1:
        differences = np.diff(inertias)
        second_diff = np.diff(differences)
        if len(second_diff) > 0:
            optimal_k = k_range[np.argmax(second_diff) + 1] if len(second_diff) > 0 else 3
        else:
            optimal_k = 3
    else:
        optimal_k = 3
        
    return optimal_k

## 4. Perform Clustering on All Embeddings

In [ ]:
print("🔍 Performing clustering analysis...")
results_dict = {}
all_results = []

for data in embeddings_data:
    dataset = data['dataset']
    dim = data['dimension']
    embeddings = data['embeddings']
    true_labels = data['true_labels']
    
    print(f"\n📊 Analyzing {dataset}/{dim}...")
    
    clustering_results, embeddings_scaled = perform_clustering(embeddings, true_labels)
    results_dict[f"{dataset}_{dim}"] = (clustering_results, embeddings_scaled)
    
    # Store results
    for method in ['kmeans', 'spectral']:
        result = {
            'dataset': dataset,
            'dimension': dim,
            'algorithm': method,
            'n_clusters': len(np.unique(clustering_results[method]['labels'])),
            'silhouette_score': clustering_results[method]['silhouette'],
            'n_samples': len(embeddings),
            'embedding_dim': embeddings.shape[1]
        }
        
        if true_labels is not None:
            result['ari'] = clustering_results[method]['ari']
            result['nmi'] = clustering_results[method]['nmi']
        
        all_results.append(result)

# Create results DataFrame
results_df = pd.DataFrame(all_results)
print(f"\n✅ Clustering completed for all {len(embeddings_data)} embedding sets")

🔍 Performing clustering analysis...

✅ Clustering completed for all 0 embedding sets


## 5. Display Initial Results

In [ ]:
print("📈 Initial Clustering Results:")
display(results_df.head(10))

# Summary statistics
print("\n📊 Summary Statistics:")
if 'ari' in results_df.columns:
    summary_stats = results_df.groupby(['dataset', 'algorithm'])[['ari', 'silhouette_score']].mean()
else:
    summary_stats = results_df.groupby(['dataset', 'algorithm'])[['silhouette_score']].mean()
display(summary_stats)

📈 Initial Clustering Results:


""



📊 Summary Statistics:


KeyError: 'dataset'

## 6. Create Visualizations

In [ ]:
def create_visualizations(embeddings_data, results_dict):
    """Create t-SNE and UMAP visualizations for all embeddings"""
    plots_dir = './plots'
    os.makedirs(plots_dir, exist_ok=True)
    
    for data in embeddings_data:
        dataset = data['dataset']
        dim = data['dimension']
        embeddings = data['embeddings']
        true_labels = data['true_labels']
        
        print(f"🎨 Creating visualizations for {dataset}/{dim}...")
        
        # Get clustering results
        cluster_key = f"{dataset}_{dim}"
        if cluster_key not in results_dict:
            continue
            
        clustering_results, embeddings_scaled = results_dict[cluster_key]
        
        # Create t-SNE and UMAP projections
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
        umap_reducer = umap.UMAP(random_state=42)
        
        embeddings_tsne = tsne.fit_transform(embeddings_scaled)
        embeddings_umap = umap_reducer.fit_transform(embeddings_scaled)
        
        # Create comprehensive visualization
        create_comparison_plot(dataset, dim, embeddings_tsne, embeddings_umap, 
                             true_labels, clustering_results, plots_dir)

In [ ]:
def create_comparison_plot(dataset, dim, tsne_emb, umap_emb, true_labels, 
                          clustering_results, plots_dir):
    """Create comparison plot with true labels and clustering results"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Clustering Analysis: {dataset} - {dim}\n', fontsize=16, fontweight='bold')
    
    # Plot configurations
    plots_config = [
        (tsne_emb, 't-SNE Visualization'),
        (umap_emb, 'UMAP Visualization')
    ]
    
    for row, (embeddings_2d, title_base) in enumerate(plots_config):
        # True labels (if available)
        ax = axes[row, 0]
        if true_labels is not None:
            scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                               c=true_labels, cmap='tab10', s=30, alpha=0.7)
            ax.set_title(f'{title_base}\nTrue Labels\n(ARI Reference)')
            plt.colorbar(scatter, ax=ax)
        else:
            ax.text(0.5, 0.5, 'True Labels\nNot Available', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{title_base}\nTrue Labels')
        
        # K-means results
        ax = axes[row, 1]
        kmeans_labels = clustering_results['kmeans']['labels']
        scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                           c=kmeans_labels, cmap='tab10', s=30, alpha=0.7)
        ari = clustering_results['kmeans'].get('ari', 'N/A')
        silhouette = clustering_results['kmeans']['silhouette']
        ax.set_title(f'{title_base}\nK-means Clustering\nARI: {ari:.3f}, Silhouette: {silhouette:.3f}')
        plt.colorbar(scatter, ax=ax)
        
        # Spectral clustering results
        ax = axes[row, 2]
        spectral_labels = clustering_results['spectral']['labels']
        scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                           c=spectral_labels, cmap='tab10', s=30, alpha=0.7)
        ari = clustering_results['spectral'].get('ari', 'N/A')
        silhouette = clustering_results['spectral']['silhouette']
        ax.set_title(f'{title_base}\nSpectral Clustering\nARI: {ari:.3f}, Silhouette: {silhouette:.3f}')
        plt.colorbar(scatter, ax=ax)
    
    plt.tight_layout()
    filename = f"{plots_dir}/clustering_{dataset}_{dim}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print(f"💾 Saved visualization: {filename}")

# %%
print("🎨 Generating visualizations...")
create_visualizations(embeddings_data, results_dict)

## 7. Performance Analysis and Comparison

In [ ]:
def generate_summary_analysis(results_df):
    """Generate comprehensive summary analysis"""
    print("="*80)
    print("📊 CLUSTERING ANALYSIS SUMMARY")
    print("="*80)
    
    # Best performing embeddings by ARI (if available)
    if 'ari' in results_df.columns:
        print("\n🏆 TOP PERFORMING EMBEDDINGS (by ARI):")
        ari_results = results_df[results_df['ari'].notna()]
        best_ari = ari_results.loc[ari_results.groupby(['dataset', 'algorithm'])['ari'].idxmax()]
        
        for _, row in best_ari.iterrows():
            print(f"{row['dataset']:12} | {row['algorithm']:15} | {row['dimension']:8} | "
                  f"ARI: {row['ari']:.3f} | Silhouette: {row['silhouette_score']:.3f}")
    
    # Best by silhouette score
    print("\n🏆 TOP PERFORMING EMBEDDINGS (by Silhouette Score):")
    best_silhouette = results_df.loc[results_df.groupby(['dataset', 'algorithm'])['silhouette_score'].idxmax()]
    
    for _, row in best_silhouette.iterrows():
        ari_info = f" | ARI: {row['ari']:.3f}" if 'ari' in row and pd.notna(row['ari']) else ""
        print(f"{row['dataset']:12} | {row['algorithm']:15} | {row['dimension']:8} | "
              f"Silhouette: {row['silhouette_score']:.3f}{ari_info}")
    
    # Overall best embeddings
    print("\n🎯 OVERALL BEST EMBEDDINGS:")
    if 'ari' in results_df.columns and not results_df['ari'].isna().all():
        overall_best = results_df.loc[results_df['ari'].idxmax()]
        metric_used = "ARI"
    else:
        overall_best = results_df.loc[results_df['silhouette_score'].idxmax()]
        metric_used = "Silhouette Score"
    
    if overall_best is not None:
        print(f"Dataset: {overall_best['dataset']}")
        print(f"Dimension: {overall_best['dimension']}")
        print(f"Algorithm: {overall_best['algorithm']}")
        print(f"Silhouette Score: {overall_best['silhouette_score']:.3f}")
        if 'ari' in overall_best and pd.notna(overall_best['ari']):
            print(f"ARI: {overall_best['ari']:.3f}")
        print(f"Based on: {metric_used}")

In [ ]:
generate_summary_analysis(results_df)

## 8. Performance Heatmaps

In [ ]:
def create_performance_heatmaps(results_df):
    """Create performance heatmaps for easy comparison"""
    plots_dir = './plots'
    
    # Determine which metric to use for visualization
    if 'ari' in results_df.columns and not results_df['ari'].isna().all():
        metric = 'ari'
        title_suffix = 'ARI'
        cmap = 'viridis'
    else:
        metric = 'silhouette_score'
        title_suffix = 'Silhouette Score'
        cmap = 'coolwarm'
    
    # Create pivot table for heatmap
    pivot_data = results_df.pivot_table(
        index=['dataset', 'dimension'], 
        columns='algorithm', 
        values=metric, 
        aggfunc='mean'
    )
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_data, annot=True, cmap=cmap, fmt='.3f', 
                linewidths=0.5, cbar_kws={'label': title_suffix})
    plt.title(f'Clustering Performance ({title_suffix}) by Dataset and Algorithm\n(Higher is Better)')
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/clustering_performance_heatmap.png', 
               dpi=300, bbox_inches='tight')
    plt.show()
    
    # Algorithm comparison boxplot
    plt.figure(figsize=(10, 6))
    if metric == 'ari':
        sns.boxplot(data=results_df, x='algorithm', y='ari')
        plt.title('ARI Distribution by Clustering Algorithm')
        plt.ylabel('Adjusted Rand Index (ARI)')
    else:
        sns.boxplot(data=results_df, x='algorithm', y='silhouette_score')
        plt.title('Silhouette Score Distribution by Clustering Algorithm')
        plt.ylabel('Silhouette Score')
    
    plt.xlabel('Clustering Algorithm')
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/algorithm_comparison.png', 
               dpi=300, bbox_inches='tight')
    plt.show()
    
    # Dimension comparison
    plt.figure(figsize=(10, 6))
    if metric == 'ari':
        sns.boxplot(data=results_df, x='dimension', y='ari', hue='algorithm')
        plt.title('ARI by Embedding Dimension and Algorithm')
        plt.ylabel('Adjusted Rand Index (ARI)')
    else:
        sns.boxplot(data=results_df, x='dimension', y='silhouette_score', hue='algorithm')
        plt.title('Silhouette Score by Embedding Dimension and Algorithm')
        plt.ylabel('Silhouette Score')
    
    plt.xlabel('Embedding Dimension')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/dimension_comparison.png', 
               dpi=300, bbox_inches='tight')
    plt.show()

# %%


In [ ]:
print("📊 Creating performance heatmaps...")
create_performance_heatmaps(results_df)

## 9. Answer Key Questions

In [ ]:
def answer_research_questions(results_df):
    """Answer the key research questions"""
    print("="*80)
    print("🔬 RESEARCH QUESTIONS ANALYSIS")
    print("="*80)
    
    # Question 1: Which embeddings yield the clearest cluster separation?
    print("\n1. WHICH EMBEDDINGS YIELD THE CLEAREST CLUSTER SEPARATION?")
    
    if 'ari' in results_df.columns:
        best_ari = results_df.loc[results_df.groupby('dataset')['ari'].idxmax()]
        print("Based on ARI (higher is better):")
        for _, row in best_ari.iterrows():
            print(f"   {row['dataset']}: {row['dimension']} with {row['algorithm']} "
                  f"(ARI: {row['ari']:.3f}, Silhouette: {row['silhouette_score']:.3f})")
    
    best_silhouette = results_df.loc[results_df.groupby('dataset')['silhouette_score'].idxmax()]
    print("\nBased on Silhouette Score (higher is better):")
    for _, row in best_silhouette.iterrows():
        ari_info = f", ARI: {row['ari']:.3f}" if 'ari' in row and pd.notna(row['ari']) else ""
        print(f"   {row['dataset']}: {row['dimension']} with {row['algorithm']} "
              f"(Silhouette: {row['silhouette_score']:.3f}{ari_info})")
    
    # Question 2: Which algorithm performs better?
    print("\n2. WHICH CLUSTERING ALGORITHM PERFORMS BETTER?")
    algo_performance = results_df.groupby('algorithm').agg({
        'silhouette_score': ['mean', 'std'],
        'ari': ['mean', 'std'] if 'ari' in results_df.columns else None
    }).round(3)
    
    display(algo_performance)
    
    # Question 3: Does embedding dimension affect clustering quality?
    print("\n3. DOES EMBEDDING DIMENSION AFFECT CLUSTERING QUALITY?")
    dim_performance = results_df.groupby('dimension').agg({
        'silhouette_score': ['mean', 'std'],
        'ari': ['mean', 'std'] if 'ari' in results_df.columns else None
    }).round(3)
    
    display(dim_performance)

In [ ]:
answer_research_questions(results_df)

## 10. Save Results

In [ ]:
# Save detailed results to CSV
results_df.to_csv('clustering_analysis_results.csv', index=False)
print("💾 Results saved to 'clustering_analysis_results.csv'")

In [ ]:
print("\n" + "="*80)
print("✅ ANALYSIS COMPLETED SUCCESSFULLY!")
print("="*80)
print("\n📁 Generated Files:")
print("   - clustering_analysis_results.csv (detailed results)")
print("   - plots/clustering_*.png (individual visualizations)")
print("   - plots/clustering_performance_heatmap.png")
print("   - plots/algorithm_comparison.png")
print("   - plots/dimension_comparison.png")
print("\n🎯 Next Steps:")
print("   - Examine the visualizations in the 'plots' folder")
print("   - Check which embeddings/dimensions work best for your datasets")
print("   - Consider the trade-off between ARI and Silhouette scores")

## 11. Final Summary Table

In [ ]:
# Create a final summary table for quick reference
if 'ari' in results_df.columns:
    summary_table = results_df.pivot_table(
        index=['dataset', 'dimension'],
        columns='algorithm',
        values=['ari', 'silhouette_score'],
        aggfunc='mean'
    ).round(3)
else:
    summary_table = results_df.pivot_table(
        index=['dataset', 'dimension'],
        columns='algorithm',
        values='silhouette_score',
        aggfunc='mean'
    ).round(3)

print("📋 FINAL SUMMARY TABLE:")
display(summary_table)